# Olist E-Commerce Marketplace — Business Analysis

**Project 1 | Python / Pandas Analysis**

This notebook supports the Olist Brazilian e-commerce marketplace analysis. It focuses on the strongest analytical work used in the final project story and is organized for portfolio publication.

### Business objective

Understand what is happening to marketplace growth and identify where customer, category, operational, and seller performance require management attention.

### Core questions

1. Is the 2018 slowdown primarily a volume or basket-value problem?
2. How strong is observed repeat purchasing?
3. Which categories are under pressure or expanding?
4. Where do commercial and operational risks overlap?
5. How concentrated is marketplace activity among sellers?

> **Note:** The original working notebook is preserved separately. This notebook is the cleaned publication version.

## 1. Setup & Data Loading

The raw Olist CSV files should be placed in a `data/` folder when this notebook is published to GitHub.

The fallback to the current directory allows the notebook to remain usable with the original local project folder.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data")
if not DATA_DIR.exists():
    DATA_DIR = Path(".")

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")

print("Orders:", orders.shape)
print("Order items:", order_items.shape)
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)

In [ ]:
# Convert timestamp fields once, near the beginning of the notebook.
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders["order_month"] = orders["order_purchase_timestamp"].dt.to_period("M")

delivered_orders = orders.loc[orders["order_status"] == "delivered"].copy()

# Delivery duration in days.
delivered_orders["delivery_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Difference between actual delivery and estimated delivery.
delivered_orders["delivery_variance_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

print("Delivered orders:", len(delivered_orders))
print("Missing delivery-duration values:", delivered_orders["delivery_days"].isna().sum())

## 2. Delivery Performance

Delivery performance is treated as an operational indicator rather than a causal explanation for sales or review outcomes.

The dataset contains a small number of delivered orders without a customer-delivery timestamp, so delivery-duration calculations use the available observations.

In [ ]:
delivery_summary = delivered_orders["delivery_days"].describe()
delivery_summary

In [ ]:
delivery_metrics = pd.Series({
    "delivered_orders": len(delivered_orders),
    "usable_delivery_durations": delivered_orders["delivery_days"].notna().sum(),
    "missing_delivery_durations": delivered_orders["delivery_days"].isna().sum(),
    "mean_delivery_days": delivered_orders["delivery_days"].mean(),
    "median_delivery_days": delivered_orders["delivery_days"].median(),
})
delivery_metrics

In [ ]:
delivered_orders["delivery_status"] = np.select(
    [
        delivered_orders["delivery_variance_days"].isna(),
        delivered_orders["delivery_variance_days"] < 0,
        delivered_orders["delivery_variance_days"] == 0,
        delivered_orders["delivery_variance_days"] > 0,
    ],
    [
        "Missing estimate/actual",
        "Early",
        "On time",
        "Late",
    ],
    default="Unknown",
)

delivery_status_share = (
    delivered_orders["delivery_status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

delivery_status_share

## 3. Sales & Marketplace Growth

Sales are measured using item-level `price` for delivered orders. The main comparable growth window is **January to August 2018**, avoiding the partial first and last months of the full dataset.

In [ ]:
delivered_sales = (
    order_items
    .merge(
        orders[["order_id", "order_status", "order_purchase_timestamp", "order_month"]],
        on="order_id",
        how="left",
    )
    .loc[lambda df: df["order_status"] == "delivered"]
    .copy()
)

total_sales = delivered_sales["price"].sum()

monthly_sales = delivered_sales.groupby("order_month")["price"].sum()
monthly_orders = delivered_sales.groupby("order_month")["order_id"].nunique()
monthly_aov = monthly_sales / monthly_orders

sales_kpis = pd.Series({
    "total_delivered_sales": total_sales,
    "total_delivered_orders": delivered_sales["order_id"].nunique(),
    "total_customers": customers["customer_unique_id"].nunique(),
    "overall_aov": total_sales / delivered_sales["order_id"].nunique(),
})

sales_kpis

In [ ]:
jan_sales = monthly_sales.loc[pd.Period("2018-01")]
aug_sales = monthly_sales.loc[pd.Period("2018-08")]
jan_orders = monthly_orders.loc[pd.Period("2018-01")]
aug_orders = monthly_orders.loc[pd.Period("2018-08")]
jan_aov = monthly_aov.loc[pd.Period("2018-01")]
aug_aov = monthly_aov.loc[pd.Period("2018-08")]

jan_aug_growth = pd.Series({
    "Sales growth": (aug_sales / jan_sales - 1) * 100,
    "Order growth": (aug_orders / jan_orders - 1) * 100,
    "AOV growth": (aug_aov / jan_aov - 1) * 100,
}).round(2)

jan_aug_growth

### Finding — Growth slowdown

Between January and August 2018:

- Orders declined by approximately **10.16%**
- Sales declined by approximately **9.31%**
- AOV increased by approximately **0.94%**

This indicates that the slowdown was primarily associated with lower transaction volume rather than a meaningful deterioration in average order value.

## 4. Customer Behavior & Repeat Purchasing

Customer identity is evaluated using `customer_unique_id`, rather than `customer_id`, because a unique customer can appear across multiple order records.

In [ ]:
customer_order_history = (
    orders[["order_id", "customer_id", "order_purchase_timestamp", "order_month", "order_status"]]
    .merge(
        customers[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="left",
    )
)

customer_order_counts = (
    customer_order_history
    .groupby("customer_unique_id")["order_id"]
    .nunique()
)

customer_segments = pd.Series({
    "One-time observed customers": (customer_order_counts == 1).sum(),
    "Repeat observed customers": (customer_order_counts > 1).sum(),
    "Total observed customers": customer_order_counts.size,
})

customer_segments["One-time share %"] = (
    customer_segments["One-time observed customers"]
    / customer_segments["Total observed customers"] * 100
)
customer_segments["Repeat share %"] = (
    customer_segments["Repeat observed customers"]
    / customer_segments["Total observed customers"] * 100
)

customer_segments.round(2)

In [ ]:
delivered_sales_customer = delivered_sales.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left",
)

repeat_customer_ids = customer_order_counts[customer_order_counts > 1].index

repeat_revenue = delivered_sales_customer.loc[
    delivered_sales_customer["customer_unique_id"].isin(repeat_customer_ids),
    "price"
].sum()

repeat_revenue_share = repeat_revenue / total_sales * 100

pd.Series({
    "Repeat-customer revenue": repeat_revenue,
    "Repeat-customer revenue share %": repeat_revenue_share,
})

In [ ]:
# Returning-customer trend: customers whose first observed purchase occurred before the current month.
first_purchase = (
    customer_order_history
    .groupby("customer_unique_id")["order_month"]
    .min()
)

customer_order_history["customer_type"] = np.where(
    customer_order_history["order_month"]
    == customer_order_history["customer_unique_id"].map(first_purchase),
    "New",
    "Returning",
)

monthly_customer_type = (
    customer_order_history
    .groupby(["order_month", "customer_type"])["customer_unique_id"]
    .nunique()
    .unstack(fill_value=0)
)

returning_jan = monthly_customer_type.loc[pd.Period("2018-01"), "Returning"]
returning_aug = monthly_customer_type.loc[pd.Period("2018-08"), "Returning"]

returning_customer_growth = (returning_aug / returning_jan - 1) * 100

pd.Series({
    "January 2018 returning customers": returning_jan,
    "August 2018 returning customers": returning_aug,
    "Returning-customer growth %": returning_customer_growth,
}).round(2)

### Finding — Repeat purchasing

Only about **3.12% of observed customers placed more than one order** during the dataset observation period, while repeat customers generated approximately **R$0.78M / 5.73% of sales**.

Returning customers also increased by roughly **34% from January to August 2018**.

**Important limitation:** a "one-time observed customer" is not confirmed permanent churn. Customers who first purchased near the end of the observation period had less opportunity to return.

## 5. Category Performance

Category analysis compares January and August 2018 sales and order volume. Categories are then classified according to the direction of order volume and sales per order.

In [ ]:
sales_product = delivered_sales.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left",
)

category_driver = (
    sales_product
    .groupby(["order_month", "product_category_name"])
    .agg(
        sales=("price", "sum"),
        orders=("order_id", "nunique"),
    )
)

category_driver["sales_per_order"] = (
    category_driver["sales"] / category_driver["orders"]
)

jan = pd.Period("2018-01")
aug = pd.Period("2018-08")

category_driver_2018 = pd.DataFrame({
    "jan_sales": category_driver["sales"].unstack("order_month")[jan],
    "aug_sales": category_driver["sales"].unstack("order_month")[aug],
    "jan_orders": category_driver["orders"].unstack("order_month")[jan],
    "aug_orders": category_driver["orders"].unstack("order_month")[aug],
    "jan_sales_per_order": category_driver["sales_per_order"].unstack("order_month")[jan],
    "aug_sales_per_order": category_driver["sales_per_order"].unstack("order_month")[aug],
})

category_driver_2018["sales_change"] = (
    category_driver_2018["aug_sales"] - category_driver_2018["jan_sales"]
)
category_driver_2018["sales_change_pct"] = (
    category_driver_2018["sales_change"]
    / category_driver_2018["jan_sales"] * 100
)

In [ ]:
def classify_driver(row):
    if pd.isna(row["jan_orders"]) or pd.isna(row["jan_sales_per_order"]):
        return "New / No January activity"
    if row["aug_orders"] < row["jan_orders"] and row["aug_sales_per_order"] < row["jan_sales_per_order"]:
        return "Double pressure"
    if row["aug_orders"] < row["jan_orders"] and row["aug_sales_per_order"] > row["jan_sales_per_order"]:
        return "Volume-driven"
    if row["aug_orders"] > row["jan_orders"] and row["aug_sales_per_order"] < row["jan_sales_per_order"]:
        return "Value/mix pressure"
    if row["aug_orders"] > row["jan_orders"] and row["aug_sales_per_order"] > row["jan_sales_per_order"]:
        return "Expansion"
    return "No material change"

category_driver_2018["driver"] = category_driver_2018.apply(classify_driver, axis=1)

driver_summary = (
    category_driver_2018["driver"]
    .value_counts()
    .rename("categories")
    .to_frame()
)

driver_summary

In [ ]:
double_pressure = category_driver_2018[
    category_driver_2018["driver"] == "Double pressure"
].copy()

double_pressure_sales = pd.Series({
    "January sales": double_pressure["jan_sales"].sum(),
    "August sales": double_pressure["aug_sales"].sum(),
    "Sales change": double_pressure["sales_change"].sum(),
})

double_pressure_sales["Sales change %"] = (
    double_pressure_sales["August sales"]
    / double_pressure_sales["January sales"] - 1
) * 100

double_pressure_sales.round(2)

### Finding — Category polarization

Across **73 categories**, performance split into six growth patterns:

- 18 Volume-driven
- 17 Double Pressure
- 16 Value/Mix Pressure
- 10 Expansion
- 7 New / No January activity
- 5 No Material Change

The Double Pressure group declined from approximately **R$371.3K to R$201.6K**, or about **45.7%**.

The implication is that the marketplace slowdown was not uniform across the assortment.

## 6. Commercial + Operational Multi-Risk Categories

This section identifies high-sales categories where sales performance overlaps with freight intensity and late-delivery risk.

This is a **risk-screening analysis**, not a causal model. The presence of high freight or late delivery does not prove that either caused the sales decline.

In [ ]:
# Category-level sales and freight.
category_economics = (
    sales_product
    .groupby("product_category_name")
    .agg(
        sales=("price", "sum"),
        freight=("freight_value", "sum"),
    )
)

category_economics["freight_to_sales_pct"] = (
    category_economics["freight"]
    / category_economics["sales"] * 100
)

# Build an order-category table so delivery is counted at order level.
category_order_values = (
    sales_product
    .groupby(["product_category_name", "order_id"], as_index=False)
    .agg(
        category_sales=("price", "sum"),
        category_freight=("freight_value", "sum"),
    )
    .merge(
        delivered_orders[["order_id", "delivery_variance_days"]],
        on="order_id",
        how="left",
    )
)

category_late = (
    category_order_values
    .assign(is_late=lambda df: df["delivery_variance_days"] > 0)
    .groupby("product_category_name")
    .agg(
        category_orders=("order_id", "nunique"),
        late_orders=("is_late", "sum"),
    )
)

category_late["late_rate"] = (
    category_late["late_orders"]
    / category_late["category_orders"] * 100
)

multi_risk = (
    category_driver_2018[["jan_sales", "aug_sales", "sales_change", "sales_change_pct"]]
    .join(category_economics[["freight_to_sales_pct"]], how="left")
    .join(category_late[["category_orders", "late_rate"]], how="left")
)

# Focus on commercially meaningful categories.
multi_risk_high_sales = (
    multi_risk[multi_risk["jan_sales"] + multi_risk["aug_sales"] >= 100000]
    .sort_values(["sales_change", "late_rate"], ascending=[True, False])
)

multi_risk_high_sales

### Example high-sales categories under combined pressure

The screening highlights categories such as **eletronicos**, **moveis_escritorio**, **moveis_decoracao**, and **cama_mesa_banho**, where declining sales overlap with relatively high freight intensity and late-delivery rates.

These categories should be investigated further across seller mix, fulfillment, pricing, shipping economics, and customer experience.

## 7. Seller Concentration

Seller concentration is measured using delivered sales. The objective is to understand whether marketplace performance is dominated by a small number of sellers.

In [ ]:
seller_sales = (
    delivered_sales
    .groupby("seller_id")
    .agg(
        sales=("price", "sum"),
        orders=("order_id", "nunique"),
        items=("order_item_id", "count"),
    )
    .sort_values("sales", ascending=False)
)

seller_sales["sales_share"] = (
    seller_sales["sales"] / seller_sales["sales"].sum() * 100
)

seller_concentration = pd.Series({
    "Top 10 seller sales share %": seller_sales.head(10)["sales_share"].sum(),
    "Top 20 seller sales share %": seller_sales.head(20)["sales_share"].sum(),
    "Top 50 seller sales share %": seller_sales.head(50)["sales_share"].sum(),
    "Largest seller sales share %": seller_sales.iloc[0]["sales_share"],
})

seller_concentration.round(2)

### Finding — Seller concentration

The leading sellers account for a meaningful but not overwhelming share of delivered sales:

- Top 10: **13.27%**
- Top 20: **21.28%**
- Top 50: **33.20%**
- Largest seller: **1.72%**

This suggests a broad seller base, while still making high-value sellers important for marketplace performance and operational resilience.

## 8. Supporting Operational Analysis

The project also examined customer geography and seller/destination delivery patterns. These analyses are retained as supporting diagnostics rather than headline findings.

They can help management investigate whether operational performance varies materially by destination or seller segment.

In [ ]:
customer_location = customers[["customer_id", "customer_state", "customer_city"]]

delivery_geo = delivered_orders.merge(
    customer_location,
    on="customer_id",
    how="left",
)

delivery_geo["is_late"] = delivery_geo["delivery_variance_days"] > 0

state_delivery = (
    delivery_geo
    .groupby("customer_state")
    .agg(
        total_orders=("order_id", "nunique"),
        late_orders=("is_late", "sum"),
    )
)

state_delivery["late_rate"] = (
    state_delivery["late_orders"]
    / state_delivery["total_orders"] * 100
)

state_delivery[state_delivery["total_orders"] >= 1000].sort_values(
    "late_rate", ascending=False
)

In [ ]:
economic_summary = delivered_sales[["price", "freight_value"]].sum()
economic_summary["freight_to_sales_pct"] = (
    economic_summary["freight_value"]
    / economic_summary["price"] * 100
)

economic_summary

## 9. Final Analytical Takeaways

### 1. Growth slowdown
The Jan–Aug 2018 slowdown was primarily associated with declining transaction volume: orders fell **10.16%**, sales fell **9.31%**, while AOV increased **0.94%**.

### 2. Customer retention
Only **3.12%** of observed customers placed more than one order, but repeat customers contributed approximately **5.73% of sales** and returning customers increased by roughly **34%** over Jan–Aug 2018.

### 3. Category polarization
Performance varied substantially across categories. The **Double Pressure** group experienced a roughly **45.7% sales decline**, indicating concentrated rather than uniform pressure.

### 4. Multi-risk categories
Several high-sales categories showed simultaneous sales deterioration, freight intensity, and elevated late-delivery rates. These are priority areas for deeper operational investigation.

### 5. Seller concentration
The top 50 sellers represented approximately **33.20% of delivered sales**, while the largest seller represented only **1.72%**, indicating broad marketplace participation with meaningful concentration among leading sellers.

---

## 10. Limitations

- The dataset represents a historical observation period.
- The first and final months are partial; Jan–Aug 2018 was used for the main comparable growth analysis.
- A one-time observed customer is not necessarily a permanently churned customer.
- Observational analysis does not establish causality.
- Freight-to-sales ratios should not be interpreted as seller profitability.
- Multi-risk categories identify areas for investigation, not proven causes of sales decline.
- Implementation decisions would require additional operational, customer, and seller-level data.

## Project Role of Python

Python was used to perform exploratory analysis, calculate business metrics, classify category performance, investigate customer and seller behavior, and cross-check important results used in the final SQL and Power BI deliverables.

**Final workflow:**

`Raw Data → Python/Pandas → SQL → Power BI → Business Diagnosis → Recommendations`